In [ ]:


# --- Step 2: Install required libraries ---
!pip install imdlib xarray netCDF4

import imdlib as imd
import xarray as xr
import numpy as np
import os
import shutil
from google.colab import drive

# --- Step 3: Mount Google Drive ---
drive.mount('/content/drive')

# --- Step 4: Create folders ---
# Local temp folder (faster, no sync issues)
temp_folder = "/content/temp_nc"
os.makedirs(temp_folder, exist_ok=True)

# Final Google Drive folder
output_folder = "/content/drive/MyDrive/IMD_pr_1985_201new"
os.makedirs(output_folder, exist_ok=True)
print(f"✅ Temp folder: {temp_folder}")
print(f"✅ Output folder: {output_folder}")

# --- Step 5: Define the year range ---
start_year = 1985
end_year = 2014

# --- Step 6: Download and save year by year ---
successful_years = []
failed_years = []

for year in range(start_year, end_year + 1):
    print(f"\n{'='*50}")
    print(f"Processing IMD rainfall data for {year}...")

    try:
        # Download IMD gridded rainfall for the year
        data = imd.get_data('rain', year, year)

        # Convert to xarray
        ds = data.get_xarray()

        # Add metadata
        ds.attrs['title'] = f'IMD Gridded Rainfall Data - {year}'
        ds.attrs['source'] = 'India Meteorological Department'
        ds.attrs['year'] = year
        ds.attrs['crs'] = 'EPSG:4326'

        # Save to LOCAL temp folder first (avoids Google Drive sync issues)
        temp_nc = os.path.join(temp_folder, f"IMD_rainfall_{year}.nc")
        print(f"Saving to local temp: {temp_nc}")
        ds.to_netcdf(temp_nc, format='NETCDF4')

        # Copy to Google Drive
        final_nc = os.path.join(output_folder, f"IMD_rainfall_{year}.nc")
        print(f"Copying to Google Drive: {final_nc}")
        shutil.copy2(temp_nc, final_nc)

        # Remove temp file to save space
        os.remove(temp_nc)

        print(f"✅ Saved: IMD_rainfall_{year}.nc")
        print(f"   Shape: {ds['rain'].shape}")
        print(f"   Time range: {ds.time.values[0]} to {ds.time.values[-1]}")
        successful_years.append(year)

        # Clean up
        ds.close()
        del ds, data

    except Exception as e:
        print(f"❌ Error processing {year}: {str(e)}")
        print(f"   Error type: {type(e).__name__}")
        failed_years.append(year)

        # Clean up temp file if it exists
        temp_nc = os.path.join(temp_folder, f"IMD_rainfall_{year}.nc")
        if os.path.exists(temp_nc):
            os.remove(temp_nc)
        continue

# Clean up temp folder
shutil.rmtree(temp_folder, ignore_errors=True)

print(f"\n{'='*50}")
print(f"✅ Processing complete!")
print(f"Successful: {len(successful_years)} years - {successful_years}")
print(f"Failed: {len(failed_years)} years - {failed_years}")
print(f"All files saved in: {output_folder}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Temp folder: /content/temp_nc
✅ Output folder: /content/drive/MyDrive/IMD_pr_1985_201new

Processing IMD rainfall data for 1985...
Downloading: rain for year 1985
Download Successful !!!
Saving to local temp: /content/temp_nc/IMD_rainfall_1985.nc
Copying to Google Drive: /content/drive/MyDrive/IMD_pr_1985_201new/IMD_rainfall_1985.nc
✅ Saved: IMD_rainfall_1985.nc
   Shape: (365, 129, 135)
   Time range: 1985-01-01T00:00:00.000000000 to 1985-12-31T00:00:00.000000000

Processing IMD rainfall data for 1986...
Downloading: rain for year 1986
Download Successful !!!
Saving to local temp: /content/temp_nc/IMD_rainfall_1986.nc
Copying to Google Drive: /content/drive/MyDrive/IMD_pr_1985_201new/IMD_rainfall_1986.nc
✅ Saved: IMD_rainfall_1986.nc
   Shape: (365, 129, 135)
   Time range: 1986-01-01T00:00:00.000000000 to 1986-12-31T00:00:00.000000000

Processing IMD rain